# ML-08 — Capstone Modeling Lane

## 1. Method choice and why

My lane is a **"which pages first?" ranking** problem, so the toolkit entry is *any classifier's probability, evaluated at precision@K*. I start with **logistic regression** (readable, inspectable coefficients) and add **histogram gradient boosting** as a stronger non-linear comparator, to see whether complexity earns its place. Simplicity is a feature: if the models cannot clearly beat the ML-07 rule, the rule ships. The label is the observed **April-2026 forward-decline** flag; features are the same history-only warehouse vector, so nothing can read the answer key (leakage re-checked in ML-09). Seeds are fixed (`SEED = 42`) and versions printed.

In [1]:
import pandas as pd, numpy as np, sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

SEED = 42
OUT = "../outputs"
print("versions -> sklearn", sklearn.__version__, "| numpy", np.__version__, "| seed", SEED)

d = pd.read_csv(f"{OUT}/feature_vector.csv")
NUM = ["log_impressions_prev_30d","log_clicks_prev_30d","log_sessions_prev_30d","log_impr_older_30d",
       "ctr_prev_30d","hist_impr_momentum","log_search_volume","competition","cpc",
       "word_count","char_count","content_age_days","days_since_last_update","age_tier_order",
       "has_keyword_data","has_word_count"]
CAT = ["content_type","main_intent","competition_level"]
y = d["future_decline"].to_numpy()
groups = d["client_id"].to_numpy()
BASE = y.mean()
print(f"{len(y):,} pages | {d.client_id.nunique()} clients | decline base rate {BASE:.3f}")

versions -> sklearn 1.9.0 | numpy 2.4.6 | seed 42


145,030 pages | 28 clients | decline base rate 0.544


## 2. Split design

**Grouped by client, not random.** Pages from one client share hidden character (same site, templates, audience), so a random split lets the model memorize a client and fake skill. The honest question is *"does it work on a client it has never seen?"* I use **`GroupKFold(5)` with `cross_val_predict`**, so every one of the ~145k pages gets a prediction from a model that never trained on its client — full-coverage out-of-fold scores that compare head-to-head with the ML-07 baseline. The past→future structure is inside every row (features from days 31–90, label from the following 30 days), so each prediction is genuinely history→future; a calendar time-split across anchors is the natural next step on the wider panel.

## 3. Train + compare vs my baseline

In [2]:
def prec_at_k(scores, k, mask=None):
    idx = np.arange(len(y)) if mask is None else np.where(mask)[0]
    order = idx[np.argsort(-scores[idx], kind="stable")]
    return y[order[:k]].mean()

MED_CTR = d.ctr_prev_30d.median()
base = ((d.ctr_prev_30d < MED_CTR).astype(int)
        + (d.days_since_last_update >= 31).astype(int)
        + (d.hist_impr_momentum < -0.05).astype(int)).to_numpy() \
       + (0.5 - d.ctr_prev_30d.rank(pct=True).to_numpy()) * 1e-3

gkf = GroupKFold(n_splits=5)
logit = Pipeline([("pre", ColumnTransformer([("num", StandardScaler(), NUM),
                                             ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])),
                  ("clf", LogisticRegression(max_iter=2000, random_state=SEED))])
hgb = Pipeline([("pre", ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT)],
                                          remainder="passthrough")),
                ("clf", HistGradientBoostingClassifier(random_state=SEED))])
p_logit = cross_val_predict(logit, d[NUM+CAT], y, cv=gkf, groups=groups, method="predict_proba")[:, 1]
p_hgb   = cross_val_predict(hgb,   d[NUM+CAT], y, cv=gkf, groups=groups, method="predict_proba")[:, 1]

print(f"=== out-of-fold (GroupKFold=5, unseen clients), all {len(y):,} pages ===")
print(f"{'method':<24}{'AUC':>7}{'P@20':>7}{'P@50':>7}{'P@100':>8}")
print(f"{'base rate / dummy':<24}{'--':>7}{BASE:>7.3f}{BASE:>7.3f}{BASE:>8.3f}")
for n, s, a in [("rule baseline (ML-07)", base, 0), ("logistic regression", p_logit, 1),
                ("hist gradient boosting", p_hgb, 1)]:
    auc = roc_auc_score(y, s) if a else float("nan")
    print(f"{n:<24}{auc:>7.3f}{prec_at_k(s,20):>7.3f}{prec_at_k(s,50):>7.3f}{prec_at_k(s,100):>8.3f}")

mask = d.reach_impr.to_numpy() >= 100
print(f"\n=== decision-relevant slice: reach >= 100 impressions "
      f"({mask.sum():,} pages, base {y[mask].mean():.3f}) ===")
for n, s in [("rule baseline", base), ("logistic regression", p_logit), ("hist gradient boosting", p_hgb)]:
    o = np.where(mask)[0][np.argsort(-s[mask], kind="stable")][:50]
    print(f"{n:<24} P@50={prec_at_k(s,50,mask):.3f}  P@100={prec_at_k(s,100,mask):.3f}"
          f"  | top-50 median reach = {np.median(d.reach_impr.to_numpy()[o]):,.0f} impr")

d.assign(p_logit=p_logit.round(4), p_hgb=p_hgb.round(4), baseline_points=np.rint(base).astype(int))[
    ["content_id","client_id","reach_impr","p_logit","p_hgb","baseline_points","future_decline"]
].to_csv(f"{OUT}/model_scores.csv", index=False)
print(f"\nwrote {OUT}/model_scores.csv  (baseline rule is the deployable artifact; models do not beat it)")

=== out-of-fold (GroupKFold=5, unseen clients), all 145,030 pages ===
method                      AUC   P@20   P@50   P@100
base rate / dummy            --  0.544  0.544   0.544
rule baseline (ML-07)       nan  0.600  0.560   0.620
logistic regression       0.508  0.400  0.500   0.560
hist gradient boosting    0.548  0.450  0.480   0.500

=== decision-relevant slice: reach >= 100 impressions (85,410 pages, base 0.538) ===
rule baseline            P@50=0.660  P@100=0.630  | top-50 median reach = 299 impr


logistic regression      P@50=0.600  P@100=0.650  | top-50 median reach = 716 impr
hist gradient boosting   P@50=0.480  P@100=0.500  | top-50 median reach = 900 impr



wrote ../outputs/model_scores.csv  (baseline rule is the deployable artifact; models do not beat it)


**Reading the table — the honest result.** Over all pages the models barely clear chance: **AUC ≈ 0.51 (logistic) / 0.55 (HGB)**, and precision@K sits within a few points of the 0.544 base rate. On the **decision-relevant slice** (`reach ≥ 100`) the picture is blunt: the **transparent ML-07 rule (P@50 ≈ 0.66) is as good as or better than logistic (≈ 0.60) and better than HGB (≈ 0.48).**

So the model **does not beat the baseline** on the metric that matters. That is a real, publishable finding, not a failure to report around: predicting a single page's next-month decline from history alone is genuinely weak signal, and a readable three-condition rule captures most of what is there. Per the toolkit's own guidance — *simplicity is a feature* — the **deployable artifact is the ML-07 rule**; logistic is kept only as a marginal, inspectable complement. The scored file is still written for transparency and for the ML-10 comparison figure.

## 4. Errors and interpretation

In [3]:
tr, te = next(GroupShuffleSplit(1, test_size=0.3, random_state=SEED).split(d, groups=groups))
fit_l = logit.fit(d.iloc[tr][NUM+CAT], y[tr])
imp = permutation_importance(fit_l, d.iloc[te][NUM+CAT], y[te],
                             n_repeats=5, random_state=SEED, scoring="roc_auc")
print("top drivers (logistic, permutation importance = AUC drop when shuffled):")
print(pd.Series(imp.importances_mean, index=NUM+CAT).sort_values(ascending=False).head(6).round(4).to_string())

te_df = d.iloc[te].copy(); te_df["p"] = fit_l.predict_proba(d.iloc[te][NUM+CAT])[:, 1]
fp = te_df[te_df.future_decline == 0].sort_values("p", ascending=False).head(2)
fn = te_df[te_df.future_decline == 1].sort_values("p").head(1)
print("\n3 hard cases (2 false alarms, 1 miss):")
for _, r in pd.concat([fp, fn]).iterrows():
    kind = "FALSE ALARM" if r.future_decline == 0 else "MISS"
    print(f"  {kind:11s} p={r.p:.2f}  reach={r.reach_impr:>7.0f}  age={r.content_age_days:.0f}d  "
          f"days_since_update={r.days_since_last_update:.0f}  momentum={r.hist_impr_momentum:+.2f}")

top drivers (logistic, permutation importance = AUC drop when shuffled):
word_count                  0.0895
log_clicks_prev_30d         0.0419
days_since_last_update      0.0158
log_impressions_prev_30d    0.0070
has_keyword_data            0.0044
main_intent                 0.0029

3 hard cases (2 false alarms, 1 miss):
  FALSE ALARM p=0.97  reach= 134984  age=410d  days_since_update=410  momentum=-0.34
  FALSE ALARM p=0.97  reach=  11344  age=375d  days_since_update=375  momentum=+3.84
  MISS        p=0.01  reach=  36507  age=28d  days_since_update=28  momentum=+36507.00


**What it leans on.** Permutation importance (AUC drop when shuffled) puts **`word_count`** on top, then historical **clicks** and **`days_since_last_update`** (recency) — all plausible and consistent with the ML-06 audit (engagement + recency). None is *suspiciously perfect*: the largest single drop is ~0.09 AUC, exactly what a weak-but-real signal looks like, not a hidden label.

**Where it is wrong (3 hard cases).**
- **False alarm, reach ≈ 135k, old, stale:** a big, mature, un-refreshed page the model is 97% sure will slide — it held. Size and staleness are only weak risk cues.
- **False alarm, reach ≈ 11k, momentum +3.8:** a page that spiked hard and looked primed to revert — it didn't. Extreme momentum cuts both ways.
- **Miss, reach ≈ 37k, age 28d, explosive momentum:** a brand-new page ramping from near-zero (momentum blows up) that then dropped — the model read the surge as strength. New pages are their own regime.

**Takeaway.** The learned models are, at best, a **directional** echo of the ML-07 rule and do not justify replacing it. The honest recommendation is: ship the rule, gate it on reach (ML-10), and keep a human in the loop — the model's edge over a coin-flip is small and uneven.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.